# Vesti-Care — Train & Compare 3 Models (Colab GPU)

Train 3 model ML tren cung data 3-class (NORMAL / INSTABILITY / FALL):

| Cell | Model |
|------|-------|
| 8 | LSTM |
| 10 | BiLSTM |
| 12 | CNN + BiLSTM + Attention (proposed) |

**Chuan bi truoc khi chay:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload len Google Drive folder `VestiCare/data/`:
   - `X_train.npz` (420 MB)
   - `y_train.npz`
   - `X_test.npz`
   - `y_test.npz`
3. Chay tung cell theo thu tu tu tren xuong

In [ ]:
# Cell 1 — Mount Drive + kiem tra GPU
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu.returncode == 0:
    print(f'GPU: {gpu.stdout.strip()}')
else:
    print('!!! NO GPU — Runtime -> Change runtime type -> T4 GPU !!!')

In [ ]:
# Cell 2 — Duong dan (chinh neu can)
import os

DRIVE_DATA_DIR  = '/content/drive/MyDrive/VestiCare/data'
DRIVE_MODEL_DIR = '/content/drive/MyDrive/VestiCare/models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

print('Kiem tra data files:')
for f in ['X_train.npz', 'y_train.npz', 'X_test.npz', 'y_test.npz']:
    p = f'{DRIVE_DATA_DIR}/{f}'
    if os.path.exists(p):
        print(f'  [OK] {f}  ({os.path.getsize(p)/1024/1024:.1f} MB)')
    else:
        print(f'  [MISSING] {f} — upload len Drive truoc!')

In [ ]:
# Cell 3 — Hyperparameters
FOCAL_GAMMA  = 2.0
LR_INIT      = 1e-3
LR_MIN       = 1e-6
RECALL_TARGET = 0.85

CLASS_NAMES  = ['NORMAL', 'INSTABILITY', 'FALL']
NUM_CLASSES  = 3
WINDOW_SIZE  = 100
NUM_CHANNELS = 7

# Config cho tung model
CFG = {
    'lstm': {
        'epochs': 60, 'batch': 128, 'patience': 12, 'fall_w': 6.0,
        'save_as': 'fall_detection_3class_lstm.keras',
    },
    'bilstm': {
        'epochs': 80, 'batch': 128, 'patience': 15, 'fall_w': 6.0,
        'save_as': 'fall_detection_3class_bilstm.keras',
    },
    'cnn_bilstm_attn': {
        'epochs': 80, 'batch': 128, 'patience': 15, 'fall_w': 8.0,
        'save_as': 'fall_detection_3class_cnn_bilstm_attn.keras',
    },
}
print('Config OK')

In [ ]:
# Cell 4 — Import + Load data
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print(f'TF {tf.__version__}  |  GPU: {[g.name for g in tf.config.list_physical_devices("GPU")]}')

X_train = np.load(f'{DRIVE_DATA_DIR}/X_train.npz')['data'].astype(np.float32)
y_train = np.load(f'{DRIVE_DATA_DIR}/y_train.npz')['data'].astype(np.int32)
X_test  = np.load(f'{DRIVE_DATA_DIR}/X_test.npz')['data'].astype(np.float32)
y_test  = np.load(f'{DRIVE_DATA_DIR}/y_test.npz')['data'].astype(np.int32)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
for i, n in enumerate(CLASS_NAMES):
    print(f'  {n:25s}: train={int((y_train==i).sum()):6d}  test={int((y_test==i).sum()):5d}')

In [ ]:
# Cell 5 — Scaler + Augment + Normalize

def augment_minority(X, y, seed=42):
    rng = np.random.default_rng(seed)
    n0  = int((y == 0).sum())
    aug_X, aug_y = [X], [y]
    for cls in [1, 2]:
        Xc = X[y == cls]
        if len(Xc) == 0:
            continue
        rep = max(1, n0 // len(Xc) - 1)
        print(f'  {CLASS_NAMES[cls]}: {len(Xc)} windows -> x{rep+1}')
        for _ in range(rep):
            Xn = Xc.copy()
            ch = rng.integers(0, 3, size=len(Xn))
            if (ch==0).any():
                Xn[ch==0] += rng.normal(0, 0.05, Xn[ch==0].shape)
            if (ch==1).any():
                Xn[ch==1] *= rng.uniform(0.85, 1.15, (int((ch==1).sum()), 1, 1))
            if (ch==2).any():
                Xn[ch==2] = np.roll(Xn[ch==2], int(rng.integers(-3, 4)), axis=1)
            aug_X.append(Xn)
            aug_y.append(np.full(len(Xn), cls, dtype=np.int32))
    Xa = np.concatenate(aug_X)
    ya = np.concatenate(aug_y)
    p  = rng.permutation(len(Xa))
    return Xa[p], ya[p]

# Scaler tinh tu RAW X_train (truoc augment) -> nhat quan voi baseline_comparison.py
MEAN = X_train.mean(axis=(0, 1))
STD  = X_train.std(axis=(0, 1));  STD[STD == 0] = 1.0
print('Scaler (raw X_train):')
print(f'  mean: {np.round(MEAN, 4)}')
print(f'  std : {np.round(STD,  4)}')

SCALER_PATH = f'{DRIVE_MODEL_DIR}/scaler_params.npz'
np.savez(SCALER_PATH, mean=MEAN, std=STD)
print(f'  Saved: {SCALER_PATH}')

# Augment
print('\nAugmenting...')
X_aug, y_aug = augment_minority(X_train, y_train)
print(f'After augment: {len(X_aug):,} windows')

# Normalize
X_tr_n = (X_aug  - MEAN) / STD
X_te_n = (X_test - MEAN) / STD

y_tr_cat = tf.keras.utils.to_categorical(y_aug,  NUM_CLASSES)
y_te_cat = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)
print('Done.')

In [ ]:
# Cell 6 — Utilities (focal loss, train, evaluate, save)
import time
from tensorflow.keras import layers, models, regularizers

# Focal loss
def focal_loss(gamma=2.0):
    def fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        p_t    = tf.reduce_sum(y_true * y_pred, axis=-1)
        return tf.reduce_mean(((1 - p_t) ** gamma) * (-tf.math.log(p_t)))
    fn.__name__ = f'focal_g{gamma}'
    return fn

# Class weights
def make_cw(y, fall_w):
    cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y)
    cw[2] *= fall_w
    return {i: float(v) for i, v in enumerate(cw)}

# Train
def run_train(model, cfg, class_weight):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_INIT),
        loss=focal_loss(FOCAL_GAMMA),
        metrics=['accuracy'],
    )
    cbs = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=cfg['patience'],
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=cfg['patience'] // 2, min_lr=LR_MIN, verbose=1,
        ),
    ]
    t0 = time.time()
    h  = model.fit(
        X_tr_n, y_tr_cat,
        epochs=cfg['epochs'], batch_size=cfg['batch'],
        validation_data=(X_te_n, y_te_cat),
        class_weight=class_weight,
        callbacks=cbs, verbose=1,
    )
    print(f'Done in {(time.time()-t0)/60:.1f} min | best val_loss={min(h.history["val_loss"]):.4f}')
    return model

# Evaluate
def evaluate(model, name):
    t0    = time.perf_counter()
    probs = model.predict(X_te_n, batch_size=512, verbose=0)
    ms    = (time.perf_counter() - t0) * 1000 / len(y_test)

    yp = np.zeros(len(y_test), np.int32)
    yp[probs[:, 1] >= 0.50] = 1
    yp[probs[:, 2] >= 0.45] = 2

    cm   = confusion_matrix(y_test, yp, labels=[0, 1, 2])
    ftp  = float(cm[2, 2]);  ffn = float(cm[2,0]+cm[2,1]);  ffp = float(cm[0,2]+cm[1,2])
    rec  = ftp / (ftp+ffn) if (ftp+ffn) > 0 else 0
    prec = ftp / (ftp+ffp) if (ftp+ffp) > 0 else 0
    f1   = 2*rec*prec / (rec+prec) if (rec+prec) > 0 else 0
    ir   = float(cm[1,1]) / float(cm[1].sum()) if cm[1].sum() > 0 else 0
    acc  = float(np.trace(cm)) / float(cm.sum())

    tag = 'PASS' if rec >= RECALL_TARGET else f'BELOW ({rec:.4f} < {RECALL_TARGET})'
    print(f'  FALL  recall={rec:.4f}  prec={prec:.4f}  F1={f1:.4f}  FP={int(ffp)}')
    print(f'  INST  recall={ir:.4f}  |  [{tag}]')

    return dict(name=name, recall_fall=round(rec,4), prec_fall=round(prec,4),
                f1_fall=round(f1,4), recall_inst=round(ir,4),
                accuracy=round(acc,4), fp_fall=int(ffp), inf_ms=round(ms,4))

# Save model
def save_model(model, filename):
    path = f'{DRIVE_MODEL_DIR}/{filename}'
    model.save(path)
    print(f'  Saved: {filename}  ({os.path.getsize(path)/1024:.0f} KB)')
    return path

# Print comparison table
def print_table(results):
    hdr = f'{"Model":<30} {"Rec-FALL":>9} {"Prec":>7} {"F1":>7} {"Rec-INST":>9} {"FP-FALL":>8} {"ms":>7}'
    sep = '-' * len(hdr)
    print(f'\n{"="*len(hdr)}')
    print('  COMPARISON  3-class NORMAL / INSTABILITY / FALL')
    print(f'{"="*len(hdr)}')
    print(hdr); print(sep)
    for r in results:
        mk = ' <' if r['recall_fall'] >= RECALL_TARGET else ''
        print(f'  {r["name"]:<28} {r["recall_fall"]:>9.4f} {r["prec_fall"]:>7.4f} '
              f'{r["f1_fall"]:>7.4f} {r["recall_inst"]:>9.4f} {r["fp_fall"]:>8d} {r["inf_ms"]:>7.4f}{mk}')
    print(f'{"="*len(hdr)}')
    print('  < = FALL recall >= 0.85 target')

# Tracking
all_results = []
print('Utilities ready.')

---
## Model 1 — LSTM
Baseline temporal model. Bidirectional chua, chi 1 chieu.

In [ ]:
# Cell 8 — Train LSTM
cfg = CFG['lstm']

inp = layers.Input(shape=(WINDOW_SIZE, NUM_CHANNELS))
x   = layers.LSTM(64, return_sequences=True,
                  kernel_regularizer=regularizers.l2(1e-4))(inp)
x   = layers.Dropout(0.3)(x)
x   = layers.LSTM(32, kernel_regularizer=regularizers.l2(1e-4))(x)
x   = layers.Dropout(0.2)(x)
x   = layers.Dense(32, activation='relu')(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
lstm_model = models.Model(inp, out, name='lstm')
lstm_model.summary()

cw = make_cw(y_aug, cfg['fall_w'])
print(f'Class weights: {cw}')

print('\n--- Training LSTM ---')
lstm_model = run_train(lstm_model, cfg, cw)

print('\n--- Evaluate ---')
res = evaluate(lstm_model, 'LSTM')
all_results.append(res)
save_model(lstm_model, cfg['save_as'])

---
## Model 2 — BiLSTM
Bidirectional LSTM — hoc temporal pattern tu ca 2 chieu.

In [ ]:
# Cell 10 — Train BiLSTM
cfg = CFG['bilstm']

inp = layers.Input(shape=(WINDOW_SIZE, NUM_CHANNELS))
x   = layers.Bidirectional(
          layers.LSTM(64, return_sequences=True,
                      kernel_regularizer=regularizers.l2(1e-4))
      )(inp)
x   = layers.Dropout(0.3)(x)
x   = layers.Bidirectional(
          layers.LSTM(32, kernel_regularizer=regularizers.l2(1e-4))
      )(x)
x   = layers.Dropout(0.2)(x)
x   = layers.Dense(64, activation='relu')(x)
x   = layers.Dropout(0.2)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
bilstm_model = models.Model(inp, out, name='bilstm')
bilstm_model.summary()

cw = make_cw(y_aug, cfg['fall_w'])
print(f'Class weights: {cw}')

print('\n--- Training BiLSTM ---')
bilstm_model = run_train(bilstm_model, cfg, cw)

print('\n--- Evaluate ---')
res = evaluate(bilstm_model, 'BiLSTM')
all_results.append(res)
save_model(bilstm_model, cfg['save_as'])

---
## Model 3 — CNN + BiLSTM + Self-Attention (Proposed)

```
Input(100,7)
  Conv1D(64,k=5) -> BN
  Conv1D(128,k=3) -> BN -> Dropout
  Bidirectional LSTM(64, return_sequences=True)
  MultiHeadAttention(4 heads) + Residual + LayerNorm
  GlobalAvgPool -> Dense(64) -> Dense(3, softmax)
```

In [ ]:
# Cell 12 — Train CNN + BiLSTM + Attention
cfg = CFG['cnn_bilstm_attn']

inp = layers.Input(shape=(WINDOW_SIZE, NUM_CHANNELS))

# CNN block
x = layers.Conv1D(64, kernel_size=5, padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(1e-4))(inp)
x = layers.BatchNormalization()(x)
x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

# BiLSTM block
x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True,
                    kernel_regularizer=regularizers.l2(1e-4))
    )(x)
x = layers.Dropout(0.2)(x)

# Self-Attention
attn = layers.MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.1)(x, x)
x    = layers.Add()([x, attn])
x    = layers.LayerNormalization()(x)

# Head
x   = layers.GlobalAveragePooling1D()(x)
x   = layers.Dense(64, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

main_model = models.Model(inp, out, name='cnn_bilstm_attn')
main_model.summary()

cw = make_cw(y_aug, cfg['fall_w'])
print(f'Class weights: {cw}  (FALL_W={cfg["fall_w"]})')

print('\n--- Training CNN+BiLSTM+Attention ---')
main_model = run_train(main_model, cfg, cw)

print('\n--- Evaluate ---')
res = evaluate(main_model, 'CNN+BiLSTM+Attn')
all_results.append(res)
save_model(main_model, cfg['save_as'])

In [ ]:
# Cell 13 — So sanh tat ca models da train
import json

if all_results:
    print_table(all_results)
    best = max(all_results, key=lambda r: r['recall_fall'])
    print(f'\nBest FALL recall: {best["name"]}  ({best["recall_fall"]:.4f})')
    result_path = f'{DRIVE_MODEL_DIR}/deep_models_results.json'
    with open(result_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'Saved: {result_path}')
else:
    print('Khong co ket qua nao — chay Cell 8, 10, 12 truoc')

In [ ]:
# Cell 14 — Download tat ca ve may
import os
from google.colab import files

def convert_tflite(keras_path):
    tflite_path = keras_path.replace('.keras', '.tflite')
    try:
        import tensorflow as tf
        m    = tf.keras.models.load_model(keras_path, compile=False)
        conv = tf.lite.TFLiteConverter.from_keras_model(m)
        conv.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]
        conv._experimental_lower_tensor_list_ops = False
        b = conv.convert()
        with open(tflite_path, 'wb') as f:
            f.write(b)
        print(f'  TFLite: {os.path.basename(tflite_path)}  ({len(b)//1024} KB)')
    except Exception as e:
        print(f'  TFLite failed: {e}')

keras_files = [
    CFG['lstm']['save_as'],
    CFG['bilstm']['save_as'],
    CFG['cnn_bilstm_attn']['save_as'],
]

print('=== Converting TFLite ===')
for fname in keras_files:
    path = f'{DRIVE_MODEL_DIR}/{fname}'
    if os.path.exists(path):
        convert_tflite(path)
    else:
        print(f'  [SKIP] {fname} not on Drive')

print('\n=== Downloading .keras ===')
for fname in keras_files:
    path = f'{DRIVE_MODEL_DIR}/{fname}'
    if os.path.exists(path):
        print(f'  {fname}')
        files.download(path)
    else:
        print(f'  [SKIP] {fname}')

print('\n=== Downloading scaler_params.npz ===')
files.download(SCALER_PATH)

print('\n[DONE] Copy cac file sau vao ml/models_3class/:')
for fname in keras_files:
    print(f'  {fname}')
print('  scaler_params.npz  <- thay the file cu')
print('\nSau do chay: python ml/scripts/baseline_comparison.py')